<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [3]:
%%capture
!pip install transformers==4.41.2 accelerate==0.31.0

In [4]:
!nvdia-smi

/bin/bash: line 1: nvdia-smi: command not found


# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=3206


```
Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=32064, bias=False)
)
```



# Causal LM vs Prefix LM

## Causal LM
因果语言模型的核心思想是从左到右逐个预测下一个 token。"因果"这个词来源于因果关系——当前 token 的生成只能依赖于它之前（过去）的 token，不能"偷看"未来的信息。
具体来说，给定一个序列 "我 喜欢 吃 苹果"，Causal LM 的训练方式是：



*   看到 "我" → 预测 "喜欢"
*   看到 "我 喜欢" → 预测 "吃"
*   看到 "我 喜欢 吃" → 预测 "苹果"

这种限制是通过**因果注意力掩码（causal attention mask）**实现的。在注意力计算时，使用一个下三角矩阵作为掩码，把当前位置之后的所有位置遮住，使得每个 token 只能关注自己和之前的 token。假设序列有 4 个 token，掩码矩阵大致是这样的：

```
      token1    token2    token3   token4
token1  [  1      0      0      0  ]
token2  [  1      1      0      0  ]
token3  [  1      1      1      0  ]
token4  [  1      1      1      1  ]
```
1 表示可以看到，0 表示被遮住。可以看到 token3 只能关注 token1、token2 和自己，看不到 token4。

## Prefix LM

Prefix LM 是一种混合策略。它把输入序列分成两部分：前缀（prefix）和生成部分（generation）。

*   前缀部分：采用双向注意力，所有前缀 token 之间可以互相看到，类似 BERT 的行为。
*   生成部分：采用因果注意力，和 Causal LM 一样只能看到之前的内容。

举一个问答场景的例子。输入是 "问题：法国的首都是哪里？回答：" ，需要生成 "巴黎"。
在 Prefix LM 中，"问题：法国的首都是哪里？回答：" 是前缀部分，这些 token 之间可以完全双向地互相关注。而生成 "巴黎" 的时候则是从左到右因果生成。
掩码矩阵大致如下（假设前 3 个 token 是前缀，后 2 个是生成部分）：



```
        p1  p2  p3   g1  g2
  p1  [  1   1   1   0   0  ]
  p2  [  1   1   1   0   0  ]
  p3  [  1   1   1   0   0  ]
  g1  [  1   1   1   1   0  ]
  g2  [  1   1   1   1   1  ]
```
可以看到前缀部分（p1-p3）之间是全部互通的，但生成部分（g1-g2）仍然遵循因果规则。





# Causal LM 模型架构

## 嵌入层

**Token embedding**

Embedding 层本质上是一个查找表，维度为 (vocab_size, hidden_size)，即 (32064, 3072)。每个 token ID 对应表中的一行，取出一个 3072 维的向量。

```
token ID 1502  →  查表  →  [0.12, -0.34, 0.56, ..., 0.78]  (3072维)
token ID 8834  →  查表  →  [0.45, 0.23, -0.67, ..., 0.11]  (3072维)
```
如果输入序列有 L 个 token，经过 Embedding 后得到一个 (L, 3072) 的矩阵，每一行就是一个 token 的初始向量表示。

**Token Embedding 这个查找表的维度为什么是(32064, 3072)？**


详细BPE算法请见
> https://zhuanlan.zhihu.com/p/424631681

第一个维度32064

基于 BPE（Byte Pair Encoding，字节对编码）算法训练Tokenizer。首先从最小单元开始，以字节级别（256 个基础字节）或字符级别作为初始词表。然后统计训练语料中所有相邻 token 对的出现频率，将出现频率最高的一对合并成一个新 token，加入词表。不断重复这个合并过程，直到词表达到预设的目标大小。

第二个维度3072


3072 这个具体数值是模型设计者根据目标参数量反推出来的。设计一个模型时，通常先确定目标参数规模（比如 Phi-3-mini 目标约 38 亿参数），然后在几个核心超参数之间分配：

```
总参数量 ≈ 层数 × 每层参数量 + Embedding参数量

每层参数量主要由 hidden_size 决定：
  - Attention: hidden_size × 4 × hidden_size（QKV + O 投影）
  - MLP: hidden_size × 3 × intermediate_size（gate_up + down）
```






Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [ ]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Finally, we create our prompt as a user and give it to the model:

In [ ]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!
